# Notebook 02 — Retrieval Evaluation

**Goal:** Measure retrieval quality (Recall@3, MRR) and run three ablations:
- Ablation A: chunk size — 300 / 500 / 800 tokens
- Ablation B: retrieval mode — dense-only vs BM25-only vs hybrid (RRF)
- Ablation C: reranker — with and without cross-encoder

**This notebook is the differentiator.** Most portfolio RAG projects skip this entirely.

**Input:** `data/eval/test_queries.jsonl` — 30 hand-authored ground-truth queries
**Output:** `figures/02_retrieval_ablation.png`, eval metrics table

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.style.use('dark_background')

sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else str(Path.cwd()))
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# ── Load test set ────────────────────────────────────────────────────────
from src.eval import load_test_queries

queries = load_test_queries()
print(f'Total queries: {len(queries)}')
for qtype in ['in_corpus', 'borderline', 'out_of_corpus']:
    n = sum(1 for q in queries if q.get('query_type') == qtype)
    print(f'  {qtype}: {n}')

In [ ]:
# ── Ablation B: retrieval mode ─────────────────────────────────────────
from src.eval import run_retrieval_eval

results = {}
for mode, hybrid, reranker in [
    ('Dense-only',    False, False),
    ('BM25-only',     False, False),   # NOTE: modify retriever to BM25-only below
    ('Hybrid (RRF)',  True,  False),
    ('Hybrid + Rerank', True, True),
]:
    r = run_retrieval_eval(top_k=3, use_hybrid=hybrid, use_reranker=reranker, verbose=False)
    results[mode] = r
    print(f'{mode:20s}  Recall@3={r["recall_at_k"]:.3f}  MRR={r["mrr"]:.3f}')

In [ ]:
# ── Ablation A: chunk size (re-ingest with different sizes) ────────────
# Run this section ONCE per chunk size, then reset and re-run.
# For the notebook presentation, just paste the results into the summary table.

# chunk_sizes = [300, 500, 800]
# chunk_size_results = {}
# for cs in chunk_sizes:
#     from src.ingestion import ingest
#     ingest(reset=True, chunk_size=cs, overlap=cs // 10)
#     r = run_retrieval_eval(top_k=3, use_hybrid=True, use_reranker=False, verbose=False)
#     chunk_size_results[cs] = r
#     print(f'chunk_size={cs}: Recall@3={r["recall_at_k"]:.3f}  MRR={r["mrr"]:.3f}')

In [ ]:
# ── Plot: Recall@3 and MRR by retrieval mode ───────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

modes = list(results.keys())
recall = [results[m]['recall_at_k'] for m in modes]
mrr    = [results[m]['mrr'] for m in modes]

colors = ['#4f8ef7', '#f39c12', '#7dd3a8', '#e74c3c']
ax1.bar(modes, recall, color=colors)
ax1.set_title('Recall@3 by Retrieval Mode')
ax1.set_ylim(0, 1.0)
ax1.axhline(0.85, color='white', linestyle='--', alpha=0.5, label='Target 0.85')
ax1.tick_params(axis='x', rotation=20)
ax1.legend()

ax2.bar(modes, mrr, color=colors)
ax2.set_title('MRR by Retrieval Mode')
ax2.set_ylim(0, 1.0)
ax2.axhline(0.70, color='white', linestyle='--', alpha=0.5, label='Target 0.70')
ax2.tick_params(axis='x', rotation=20)
ax2.legend()

plt.tight_layout()
plt.savefig('figures/02_retrieval_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

## Retrieval Evaluation Results

| Config | Recall@3 | MRR | Notes |
|--------|----------|-----|-------|
| Dense-only | *(fill in)* | *(fill in)* | Baseline |
| BM25-only | *(fill in)* | *(fill in)* | Keyword |
| Hybrid (RRF) | *(fill in)* | *(fill in)* | |
| Hybrid + Rerank | *(fill in)* | *(fill in)* | **Final config** |

### Chunk Size Comparison

| Chunk tokens | Recall@3 | MRR |
|-------------|----------|-----|
| 300 | *(fill in)* | *(fill in)* |
| 500 | *(fill in)* | *(fill in)* |
| 800 | *(fill in)* | *(fill in)* |

**Key finding:** *(fill in after running — e.g. 'Hybrid retrieval outperforms dense-only by X MRR on engineering documents because acronyms (ASME, NPSH) are exact-match in BM25 but semantically ambiguous')*

**Next:** Notebook 03 — RAG Pipeline + Ragas Evaluation